## HUẤN LUYỆN MÔ HÌNH LLAMA-3.2-1B VỚI UNSLOTH (QLoRA)
- **Môi trường yêu cầu:** Google Colab (Runtime: T4 GPU) hoặc Máy ảo Linux có GPU NVIDIA.


### BƯỚC 1: Cài đặt thư viện (Chỉ chạy trên Google Colab)

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9pqsijkm/unsloth_6b3c32ce019d48e6af95320a58874976
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9pqsijkm/unsloth_6b3c32ce019d48e6af95320a58874976
  Resolved https://github.com/unslothai/unsloth.git to commit b24f3f61b83fab129aa67e46ff9c2d430434401a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.9 MB/s eta 0:00:0

### BƯỚC 2: Khai báo thư viện & Đăng nhập Weights & Biases
Sử dụng wandb để theo dõi biểu đồ Loss (sai số) trong quá trình train, giúp phát hiện sớm Overfitting.

In [2]:
import os

# Optional for more secure: assgin WANDB_KEY with HF_TOKEN in secret key
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_KEY", "None")
os.environ["WANDB_SILENT"] = "true"

import torch
import wandb
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### BƯỚC 3: Nạp Mô hình gốc (Base Model) & Tokenizer
Sử dụng Unsloth để nạp mô hình Llama-3.2-1B ở định dạng 4-bit, giúp tiết kiệm tối đa VRAM.

In [3]:
# --- NẠP MÔ HÌNH VÀ CẤU HÌNH TOKENIZER CHUẨN ---
max_seq_length = 2048 # Độ dài ngữ cảnh tối đa (Tương đương khoảng 1500 từ). Đủ cho RAG.
dtype = None # Unsloth tự động chọn (Bfloat16 cho Ampere+, Float16 cho T4/V100)
load_in_4bit = True # Bắt buộc True để dùng QLoRA (Tiết kiệm VRAM)

print("Đang nạp mô hình Llama-3.2-1B-Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "phamhai/Llama-3.2-1B-Instruct-Frog",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Nạp mô hình thành công!")


Đang nạp mô hình Llama-3.2-1B-Instruct...
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Unsloth: Will load phamhai/Llama-3.2-1B-Instruct-Frog as a legacy tokenizer.


phamhai/Llama-3.2-1B-Instruct-Frog does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.
Nạp mô hình thành công!


#### Kiểm tra token, ép token end và token padding phải thống nhất

In [4]:
# [QUAN TRỌNG]: ĐỒNG BỘ CẤU HÌNH TOKENIZER VÀ MODEL
# 1. Đặt Padding Side là "right" (Bắt buộc cho LLM khi train)
tokenizer.padding_side = "right"

# 2. Xử lý PAD Token: Dùng eos_token (<|eot_id|> của Llama 3) làm pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Ép cấu hình Model phải đồng nhất với Tokenizer
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print(f"Cấu hình Tokenizer: EOS ID = {tokenizer.eos_token_id}, PAD ID = {tokenizer.pad_token_id}")

Cấu hình Tokenizer: EOS ID = 128009, PAD ID = 128004


#### **Kết luận:** sau khi ép, token vẫn không thống nhất, phải kiểm tra, inspect chi tiết

In [7]:
import json

print("\n" + "="*50)
print("🔍 CẤU HÌNH MÔ HÌNH VÀ TOKENIZER")
print("="*50)

# 1. Kiểm tra các Token Đặc biệt (Special Tokens)
print("\n[1] TOKENIZER SPECIAL TOKENS:")
special_tokens = {
    "bos_token": tokenizer.bos_token,
    "bos_token_id": tokenizer.bos_token_id,
    "eos_token": tokenizer.eos_token,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token": tokenizer.pad_token,
    "pad_token_id": tokenizer.pad_token_id,
}
print(json.dumps(special_tokens, indent=4, ensure_ascii=False))



🔍 CẤU HÌNH MÔ HÌNH VÀ TOKENIZER

[1] TOKENIZER SPECIAL TOKENS:
{
    "bos_token": "<|begin_of_text|>",
    "bos_token_id": 128000,
    "eos_token": "<|eot_id|>",
    "eos_token_id": 128009,
    "pad_token": "<|finetune_right_pad_id|>",
    "pad_token_id": 128004
}


In [8]:
# 2. Kiểm tra Hành vi Tokenizer (Padding & Chat Template)
print("\n[2] TOKENIZER BEHAVIOR:")
print(f"- Padding Side (Chiều đệm): '{tokenizer.padding_side}'")
print(f"- Trạng thái Chat Template: {'✅ Đã cấu hình' if tokenizer.chat_template else '❌ BỊ THIẾU (Nguy hiểm)'}")

if tokenizer.chat_template:
    print("\n[Preview Chat Template (Bản mẫu cấu trúc prompt)]:")
    # In ra 250 ký tự đầu tiên của template để kiểm tra chuẩn Llama 3
    print(tokenizer.chat_template[:250] + "...\n")


[2] TOKENIZER BEHAVIOR:
- Padding Side (Chiều đệm): 'right'
- Trạng thái Chat Template: ✅ Đã cấu hình

[Preview Chat Template (Bản mẫu cấu trúc prompt)]:
{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- i...



In [9]:
# 3. Kiểm tra Cấu hình Lõi của Mô hình (Model Config)
print("\n[3] MODEL CONFIG HIGHLIGHTS:")
config_highlights = {
    "vocab_size": model.config.vocab_size,
    "max_position_embeddings (Context Window)": getattr(model.config, 'max_position_embeddings', 'Unknown'),
    "rope_theta (Xử lý ngữ cảnh dài)": getattr(model.config, 'rope_theta', 'Unknown'),
    "bos_token_id (Trong Model)": model.config.bos_token_id,
    "eos_token_id (Trong Model)": model.config.eos_token_id,
    "pad_token_id (Trong Model)": getattr(model.config, 'pad_token_id', 'None'),
}
print(json.dumps(config_highlights, indent=4, ensure_ascii=False))

print("="*50 + "\n")


[3] MODEL CONFIG HIGHLIGHTS:
{
    "vocab_size": 128256,
    "max_position_embeddings (Context Window)": 131072,
    "rope_theta (Xử lý ngữ cảnh dài)": "Unknown",
    "bos_token_id (Trong Model)": 128000,
    "eos_token_id (Trong Model)": 128009,
    "pad_token_id (Trong Model)": 128004
}



#### **Kết luận:** Cấu hình tokens của models và của tokenizer đã thống nhất 100%, bước tiếp theo là tiến hành huấn luyện, finetune mô hình.

### BƯỚC 4: Cấu hình QLoRA Adapters
- Không train toàn bộ 1 tỷ tham số, mà chỉ thêm vào các ma trận nhỏ (adapters).
- Rank (r) = 32 giúp mô hình học các khái niệm y khoa phức tạp.

In [10]:
# --- CẤU HÌNH QLORA ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Mức độ phức tạp của LoRA (Khuyên dùng: 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # Áp dụng lên tất cả Attention & MLP
    lora_alpha = 64, # Thường gấp đôi r
    lora_dropout = 0, # Tối ưu hóa (Unsloth khuyên dùng 0)
    bias = "none",    # Tối ưu hóa
    use_gradient_checkpointing = "unsloth", # Cực kỳ quan trọng: Giảm 30% VRAM
    random_state = 42,
    use_rslora = False,
)

Unsloth 2026.4.6 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


### BƯỚC 5: Nạp Tập dữ liệu Huấn luyện
Đọc file `final_augmented_train.jsonl` (đã được format sẵn chuẩn ChatML ở Giai đoạn 2).


In [11]:
# Tải file jsonl từ Google Drive hoặc thư mục local lên Colab
# Ví dụ trên Colab: upload file vào thư mục mặc định '/content/final_augmented_train.jsonl'
# Trong lần finetune này upload trực tiếp vào file system của máy ảo
dataset_file = "/content/final_augmented_train.jsonl"

dataset = load_dataset("json", data_files=dataset_file, split="train")
print(f"Tổng số mẫu huấn luyện: {len(dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Tổng số mẫu huấn luyện: 2302


### BƯỚC 6: Thiết lập tham số Huấn luyện (SFTTrainer)
Với dữ liệu chất lượng cao, chỉ cần 1 đến 2 epochs là đủ. Train nhiều hơn dễ gây ảo giác (overfit).


In [12]:
# --- CẤU HÌNH TRAINING ARGUMENTS ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text", # Trường chứa nội dung trong file jsonl
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set True nếu muốn train nhanh hơn cho chuỗi ngắn, False an toàn hơn cho RAG
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Effective Batch Size = 2 * 4 = 8
        warmup_steps = 50, # Khởi động learning rate từ từ
        num_train_epochs = 1, # Số vòng lặp huấn luyện. Bạn có thể tăng lên 2 nếu loss vẫn cao.
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit", # Optimizer 8-bit tiết kiệm VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none", # Báo cáo kết quả lên Weights & Biases. Tạm thời tắt ở lần này
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2302 [00:00<?, ? examples/s]

### BƯỚC 7: Bắt đầu Huấn luyện 🚀
Theo dõi cột `Loss`, nếu nó giảm dần dần từ (ví dụ) 2.0 xuống 0.7 - 0.5 là mô hình đang học rất tốt.

In [13]:
# --- BẮT ĐẦU HUẤN LUYỆN ---
print("Bắt đầu huấn luyện...")
trainer_stats = trainer.train()

Bắt đầu huấn luyện...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,302 | Num Epochs = 1 | Total steps = 288
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.195571
20,1.856705
30,1.653546
40,1.495285
50,1.476239
60,1.397594
70,1.389289
80,1.236339
90,1.270986
100,1.272679


Step,Training Loss
10,2.195571
20,1.856705
30,1.653546
40,1.495285
50,1.476239
60,1.397594
70,1.389289
80,1.236339
90,1.270986
100,1.272679


### BƯỚC 8: Kiểm thử trực tiếp Mô hình sau khi Train (Inference Test)
Hãy hỏi thử một câu RAG để xem mô hình trả lời thế nào.

In [15]:
import warnings
import transformers

# KHÓA TẤT CẢ CẢNH BÁO (WARNINGS)
# Ẩn cảnh báo FutureWarning và UserWarning của Python
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Ép Transformers chỉ in ra màn hình nếu gặp Lỗi nghiêm trọng (Error)
transformers.logging.set_verbosity_error()

# CHẠY INFERENCE
# Đặt mô hình ở chế độ suy luận (nhanh hơn 2x, giảm VRAM)
FastLanguageModel.for_inference(model)

system_prompt = "Bạn là trợ lý AI chuyên gia thú y tận tâm. Hãy trả lời câu hỏi dựa trên thông tin tham khảo được cung cấp một cách chính xác và đồng cảm. Tuyệt đối không tự bịa đặt."
context = "Bệnh Care ở chó (Canine Distemper) là bệnh truyền nhiễm nguy hiểm do virus gây ra, ảnh hưởng đến hệ hô hấp, tiêu hóa và thần kinh. Chó chưa tiêm phòng rất dễ mắc bệnh."
question = "Chó nhà tôi chưa tiêm phòng, dạo này thấy ho và tiêu chảy. Liệu có phải bệnh Care không?"

test_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

Thông tin tham khảo:
{context}

Câu hỏi: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

# Sinh câu trả lời
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
answer = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

print("\n--- CÂU TRẢ LỜI CỦA MÔ HÌNH ---")
# Cắt bỏ phần prompt ban đầu để chỉ in ra câu trả lời
print(answer.split("assistant")[-1].strip())


--- CÂU TRẢ LỜI CỦA MÔ HÌNH ---
Đừng lo lắng quá, tình trạng này có thể do virus parvo gây ra. Bạn nên đưa chó đến cơ sở thú y để được kiểm tra và điều trị kịp thời.


### BƯỚC 9: Xuất Mô hình (GGUF Export)
- Định dạng GGUF (chạy bằng llama.cpp hoặc Ollama) là đỉnh cao của sự tối ưu.
- Nó nén file chỉ còn khoảng ~1GB và có thể chạy cực mượt trên CPU hoặc VPS giá rẻ ở Giai đoạn 4.

In [16]:
# Đọc dữ liệu từ mounted google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- XUẤT MÔ HÌNH (GGUF) ---
print("Huấn luyện xong, đang xuất file GGUF...")
# Xuất định dạng lượng hóa 4-bit q4_k_m cực kỳ tối ưu cho CPU và VPS nhẹ
model.save_pretrained_gguf("Llama-3.2-1B-Pet-Chatbot-GGUF", tokenizer, quantization_method = "q4_k_m")
print("Xuất file hoàn tất!")

import shutil
import os

# Đường dẫn thư mục gốc chứa model vừa xuất
source_dir = "Llama-3.2-1B-Pet-Chatbot-GGUF"

# Đường dẫn đích trên Google Drive của bạn (có thể đổi tên thư mục Pet_Chatbot_Model tùy ý)
drive_dir = "/content/drive/MyDrive/Pet_Chatbot_Model"

print(f"Đang copy model từ {source_dir} lên Google Drive ({drive_dir})...")
shutil.copytree(source_dir, drive_dir, dirs_exist_ok=True)
print("✅ Đã sao lưu Model lên Google Drive an toàn!")

Huấn luyện xong, đang xuất file GGUF...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `Llama-3.2-1B-Pet-Chatbot-GGUF`: 100%|██████████| 1/1 [00:20<00:00, 20.72s/it]


Successfully copied all 1 files from cache to `Llama-3.2-1B-Pet-Chatbot-GGUF`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:39<00:00, 39.89s/it]


Unsloth: Merge process complete. Saved to `/content/Llama-3.2-1B-Pet-Chatbot-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
